# Wright_2017 PISN models

The four PISN models described in the paper `Neutrino signal from pair-instability supernovae' by Wright et al., Phys. Rev. D96 (2017) 103008  <https://journals.aps.org/prd/abstract/10.1103/PhysRevD.96.103008>`

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('font', size=14)

import numpy as np

from astropy import units as u 

from snewpy.flavor import ThreeFlavor
from snewpy.neutrino import MixingParameters
from snewpy.flavor_transformation import AdiabaticMSW, NoTransformation

from snewpy.models.pisn import Wright_2017
from snewpy.neutrino import MassHierarchy, MixingParameters
from snewpy.flavor_transformation import AdiabaticMSW
from snewpy.rate_calculator import RateCalculator, aggregate

## A Function which plots the Initial and Oscillated Spectra using the Adiabatic MSW Flavor Transformation and the Normal mass ordering

Plot the neutrino spectra at the source and after the requested flavor transformation has been applied.

In [ ]:
def plot_spectra(model, time=0*u.s):
    # Adiabatic MSW effect. NMO is used by default.
    xform0 = NoTransformation()
    xform_nmo = AdiabaticMSW(MixingParameters())

    # Energy array and time to compute spectra.
    # Note that any convenient units can be used and the calculation will remain internally consistent.
    E = np.linspace(0,30,151) * u.MeV

    d = 10*u.kpc  
    ispec = model.get_flux(time, E, d, xform0)    
    ospec_nmo = model.get_flux(time, E, d, xform_nmo)
        
    fig, axes = plt.subplots(1,2, figsize=(12,5), sharex=True, sharey=True, tight_layout=True)

    for i, spec in enumerate([ispec,ospec_nmo]):
        ax = axes[i]
        for flavor in ThreeFlavor:
            ax.plot(E, spec[flavor].array.squeeze() *u.erg/u.MeV, 
                   label=flavor.to_tex(),
                   color='C0' if flavor.is_electron else 'C1',
                   ls='-' if flavor.is_neutrino else ':', lw=2,
                   alpha=0.7)

        ax.set(xlabel=r'$E$ [{}]'.format(E.unit),
           title='Oscillated Flux: $t = ${:.1f}'.format(time))
        ax.grid()
        ax.legend(loc='upper right', ncol=2, fontsize=16)

        #ax.set_ylim((1e-3,1e9))        
        #ax.set_xlim((0.2,25))
        ax.set_yscale('log')
        ax.set_xscale('log')
        

    ax = axes[0]
    ax.set(ylabel=r'oscillated flux [{}]'.format(ispec[ThreeFlavor.NU_E].array.squeeze().unit))

    fig.tight_layout();

## Look at what models are available

In [ ]:
Wright_2017.param

## Go through the models, plot the original spectra and transformed spectra at $t=1$ s.

### 150$M_{\odot}$ Progenitor, Helm EOS

In [ ]:
pisn1 = Wright_2017(progenitor_mass=150*u.solMass,eos='Helm')
fig = plot_spectra(pisn1, time=-0*u.s)

### 150$M_\odot$ Progenitor, SFHo EOS

In [ ]:
pisn2 = Wright_2017(progenitor_mass=150*u.solMass,eos='SFHo')
fig = plot_spectra(pisn2, time=0*u.s)

### 250$M_\odot$ Progenitor, Helm EOS

In [ ]:
pisn3 = Wright_2017(progenitor_mass=250*u.solMass,eos='Helm')
fig = plot_spectra(pisn3, time=0*u.s)

### 250$M_\odot$ Progenitor, SFHo EOS

In [ ]:
pisn4 = Wright_2017(progenitor_mass=250*u.solMass,eos='SFHo')
fig = plot_spectra(pisn4, time=-0.15*u.s)

### Compute the total number of events in a JUNO like detector for a PISN at 10 kpc

In [ ]:
times    = pisn4.get_time()
energies = np.linspace(0,40,401)<<u.MeV
distance = 10*u.kpc

transformation = AdiabaticMSW(MixingParameters())

rc = RateCalculator()

fluence = pisn1.get_flux(t=times, E=energies, distance=distance, flavor_xform=transformation).integrate_or_sum('time')
dNdE = aggregate( rc.run(fluence, detector='scint20kt', detector_effects=False) )
Nevents = dNdE["total"].integrate_or_sum('energy').array.squeeze() 
print("150 Msun, Helm EOS", Nevents)

fluence = pisn2.get_flux(t=times, E=energies, distance=distance, flavor_xform=transformation).integrate_or_sum('time')
dNdE = aggregate( rc.run(fluence, detector='scint20kt', detector_effects=False) )
Nevents = dNdE["total"].integrate_or_sum('energy').array.squeeze() 
print("150 Msun, SFHo EOS",  Nevents)

fluence = pisn3.get_flux(t=times, E=energies, distance=distance, flavor_xform=transformation).integrate_or_sum('time')
dNdE = aggregate( rc.run(fluence, detector='scint20kt', detector_effects=False) )
Nevents = dNdE["total"].integrate_or_sum('energy').array.squeeze()
print("250 Msun, Helm EOS", Nevents)

fluence = pisn4.get_flux(t=times, E=energies, distance=distance, flavor_xform=transformation).integrate_or_sum('time')
dNdE = aggregate( rc.run(fluence, detector='scint20kt', detector_effects=False) )
Nevents = dNdE["total"].integrate_or_sum('energy').array.squeeze()
print("250 Msun, SFHo EOS", Nevents)